In [ ]:
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
import time, os
import pandas as pd
import yt_dlp


# 啟動瀏覽器工具的選項
my_options = webdriver.ChromeOptions()
# my_options.add_argument("--headless")              # 不開啟實體瀏覽器背景執行
my_options.add_argument("--start-maximized")         # 最大化視窗
my_options.add_argument("--incognito")               # 開啟無痕模式
my_options.add_argument("--disable-popup-blocking")  # 禁用彈出攔截
my_options.add_argument("--disable-notifications")   # 取消 chrome 推播通知
my_options.add_argument("--lang=zh-TW")              # 設定為正體中文

# 使用 Chrome 的 WebDriver
driver = webdriver.Chrome(
    options = my_options
)

In [ ]:
# 建立儲存圖片、影片的資料夾
folder_path = 'youtube'
os.makedirs(folder_path, exist_ok=True)

# 放置爬取的資料
li_data = []

In [ ]:
# 到 YouTube 首頁
driver.get('https://www.youtube.com/')

In [ ]:
# 輸入名稱
txtInput = driver.find_element(
    By.CSS_SELECTOR, 
    "input[name=search_query]"
)
txtInput.send_keys("chiikawa")

# 等待一下
time.sleep(1)

# 送出表單資料
txtInput.submit()

# 等待一下
time.sleep(1)

In [ ]:
# 等待篩選元素出現
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located( 
        (
            By.CSS_SELECTOR, 
            "div#filter-button"
        ) 
    )
)

#按下篩選元素，使項目浮現
driver.find_element(
    By.CSS_SELECTOR, 
    "div#filter-button"
).click()

# 等待一下
time.sleep(2)

# 按下選擇的項目
driver.find_elements(
    By.CSS_SELECTOR, 
    "a#endpoint.yt-simple-endpoint.style-scope.ytd-search-filter-renderer"
)[5].click()

# 等待一下
time.sleep(2)

In [ ]:
# 滾動到頁面底部，直到沒有新內容出現
count = 0
limit = 3
previous_height = 0

# 持續滾動頁面，直到頁面高度不再改變，且計數器達到限制次數
while count < limit:
    # 滾動到頁面底部
    driver.execute_script(
        "window.scrollTo({top: document.documentElement.scrollHeight, behavior: 'smooth'});"
    )

    # 等待頁面加載
    time.sleep(3)

    # 取得目前頁面高度
    current_height = driver.execute_script(
        "return document.documentElement.scrollHeight;"
    )

    # 如果頁面高度沒有改變，則計數器加一，直到達到限制次數
    if current_height == previous_height:
        count += 1
    else:
        previous_height = current_height
        count = 0

    # 實驗：如果頁面高度超過 600，則停止滾動
    if current_height > 600:
        break

In [ ]:
# 清空存放資料的變數
li_data.clear()

# 取得主要元素的集合
elms = driver.find_elements(
    By.CSS_SELECTOR, 
    'ytd-video-renderer.style-scope.ytd-item-section-renderer'
)

# 逐一檢視元素
for elm in elms:
    # 取得資料名稱
    a = elm.find_element(By.CSS_SELECTOR, "a#video-title")
    a_title = a.get_attribute('innerText')
    
    # 取得 YouTube 連結
    a_link = a.get_attribute('href')

    # 取得圖片連結
    img = elm.find_element(
        By.CSS_SELECTOR, 
        "a#thumbnail img"
    )
    img_src = img.get_attribute('src')
    
    # 取得 Video ID
    delimiter = '/shorts/' if '/shorts/' in a_link else 'v='
    youtube_id = a_link.split(delimiter, 1)[1] # 先取得特定分隔符號後的部分
    youtube_id = youtube_id.split('&')[0]  # 移除可能的其他參數

    # 檢視取得的資料
    print("=" * 30)
    print(a_title)
    print(a_link)
    print(img_src)
    print(youtube_id)
    
    # 將資料放入 list 中
    li_data.append({
        "id": youtube_id,
        "title": a_title,
        "link": a_link,
        "img": img_src
    })

In [ ]:
# 將資料轉換為 DataFrame 並輸出為 Excel 檔案
df = pd.DataFrame(li_data)
df.to_excel('youtube.xlsx', index=False)

In [ ]:
# 關閉瀏覽器
# 確定短時間不會執行了再關，
# 一直開開關關，可能會被對方主機認為是攻擊行為，導致封鎖 IP
driver.quit()

In [ ]:
# 下載所有 link 的影片
links = []
for index, data in enumerate(li_data):
    if index >= 3:  # 只下載前 3 個影片
        break
    
    # 取得影片連結
    links.append(data['link'])

# 下載設定
ydl_opts = {
    'outtmpl': folder_path + '/' + '%(title)s.%(ext)s',
    'noplaylist': True
}

# 批次下載影片
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download(links)